# DATA266 Task 2 — Yelp Polarity Sentiment Classification

**Member:** Anees Saheba · **Team 8**

Three models trained from scratch for binary sentiment classification. No
pretrained embeddings and no pretrained language models: every embedding table
is learned from this training split alone.

| Model | Architecture | Role |
|---|---|---|
| `baseline_max_pool` | embedding → masked max pooling → MLP | Baseline |
| `experiment_lstm` | embedding → 2-layer unidirectional LSTM → MLP | Experimental |
| `experiment_attention_pool` | embedding → learned attention pooling → MLP | Experimental |

Chosen against the team handoff (`task2_sentiment/team_handoff_yashashree.md`)
so that no architecture overlaps the other member's mean-pooling baseline,
multi-width CNN, or BiGRU.


In [ ]:
import os
import re
import csv
import json
import math
import time
import random
import platform
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score, average_precision_score,
                             matthews_corrcoef, brier_score_loss, roc_curve,
                             precision_recall_curve)

print("PyTorch:", torch.__version__)
print("Python :", platform.python_version())

In [ ]:
# ---------------------------------------------------------------------------
# Configuration, from the team handoff. Distinct from the other member's run:
# seed 4350 not 4349, vocabulary 25k not 30k, min frequency 3 not 2,
# max length 320 not 256, learning rate 2e-4 not 3e-4, 8 epochs not 5.
# ---------------------------------------------------------------------------

SEED = 4350

CONFIG = {
    "validation_fraction": 0.10,
    "vocab_size": 25_000,
    "min_frequency": 3,
    "max_length": 320,
    "batch_size": 256,
    "epochs": int(os.environ.get("DATA266_EPOCHS", 8)),
    "learning_rate": 2e-4,
    "weight_decay": 5e-4,
    "gradient_clip": 1.0,
    "seed": SEED,
}

# Subsampling for a quick pass. 0 means use the full training split.
TRAIN_SUBSET = int(os.environ.get("DATA266_TRAIN_SUBSET", 0))
TEST_SUBSET = int(os.environ.get("DATA266_TEST_SUBSET", 0))

PAD_INDEX, UNK_INDEX = 0, 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE, DEVICE_NAME = torch.device("cuda"), torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    DEVICE, DEVICE_NAME = torch.device("mps"), "Apple Silicon GPU (MPS)"
else:
    DEVICE, DEVICE_NAME = torch.device("cpu"), "CPU"

print(f"seed={SEED} device={DEVICE} ({DEVICE_NAME})")
print(json.dumps(CONFIG, indent=2))

In [ ]:
MEMBER_NAME = "anees_saheba"


def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / ".git").exists() and (candidate / "task2_sentiment").is_dir():
            return candidate
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is not None:
    MEMBER_ROOT = REPO_ROOT / "task2_sentiment" / MEMBER_NAME
    DATA_DIR = REPO_ROOT / "task2_sentiment" / "data"
    RAW_LOG_DIR = REPO_ROOT / "reproducibility" / "raw_logs"
else:
    MEMBER_ROOT = Path.cwd() / "task2_output"
    DATA_DIR = Path(os.environ.get("DATA266_YELP_DIR", "."))
    RAW_LOG_DIR = MEMBER_ROOT / "raw_logs"

DATA_DIR = Path(os.environ.get("DATA266_YELP_DIR", DATA_DIR))
CHECKPOINT_DIR = MEMBER_ROOT / "checkpoints"
OUTPUT_DIR = MEMBER_ROOT / "outputs"
PROCESSED_DIR = MEMBER_ROOT / "data_processed"
for folder in (CHECKPOINT_DIR, OUTPUT_DIR, PROCESSED_DIR, RAW_LOG_DIR,
               OUTPUT_DIR / "confusion_matrices", OUTPUT_DIR / "plots"):
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RAW_LOG_PATH = RAW_LOG_DIR / f"task2_sentiment_{RUN_ID}.log"


def log_line(message):
    """Append one timestamped line to the raw log and echo it, flushing each time."""
    stamped = f"{time.strftime('%Y-%m-%d %H:%M:%S')} {message}"
    print(stamped)
    with RAW_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(stamped + "\n")
        handle.flush()


TRAIN_CSV = DATA_DIR / "yelp_polarity_train.csv"
TEST_CSV = DATA_DIR / "yelp_polarity_test.csv"
for path in (TRAIN_CSV, TEST_CSV):
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run: bash scripts/download_data.sh task2\n"
            "or set DATA266_YELP_DIR to the directory holding the CSVs."
        )

log_line(f"run_id={RUN_ID} device={DEVICE} ({DEVICE_NAME}) seed={SEED}")
log_line("config " + " ".join(f"{k}={v}" for k, v in CONFIG.items()))
print("Raw log:", RAW_LOG_PATH)

## 1. Data preprocessing

### 1.1 Loading, malformed entries, and class balance

§2.1 requires review-length distribution, class distribution and class balance,
plus handling of missing or malformed entries. Everything below is measured
rather than assumed — class balance in particular decides whether accuracy is a
meaningful headline metric or a misleading one.


In [ ]:
train_frame = pd.read_csv(TRAIN_CSV)
test_frame = pd.read_csv(TEST_CSV)
print("raw shapes:", train_frame.shape, test_frame.shape)
print("columns   :", list(train_frame.columns))

TEXT_COLUMN = "text" if "text" in train_frame.columns else train_frame.columns[-1]
LABEL_COLUMN = "label" if "label" in train_frame.columns else train_frame.columns[0]
print(f"text column={TEXT_COLUMN!r} label column={LABEL_COLUMN!r}")


def clean_frame(frame, name):
    """Drop rows that cannot be used, reporting exactly what was removed."""
    original = len(frame)
    report = {}

    missing_text = frame[TEXT_COLUMN].isna().sum()
    missing_label = frame[LABEL_COLUMN].isna().sum()
    frame = frame.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
    report["missing_text"] = int(missing_text)
    report["missing_label"] = int(missing_label)

    frame = frame.copy()
    frame[TEXT_COLUMN] = frame[TEXT_COLUMN].astype(str)

    blank = (frame[TEXT_COLUMN].str.strip() == "").sum()
    frame = frame[frame[TEXT_COLUMN].str.strip() != ""]
    report["blank_text"] = int(blank)

    valid_labels = frame[LABEL_COLUMN].isin([0, 1])
    report["bad_label"] = int((~valid_labels).sum())
    frame = frame[valid_labels]
    frame[LABEL_COLUMN] = frame[LABEL_COLUMN].astype(int)

    duplicates = frame.duplicated(subset=[TEXT_COLUMN]).sum()
    report["duplicate_text"] = int(duplicates)

    report["rows_before"], report["rows_after"] = original, len(frame)
    print(f"{name}: {original:,} -> {len(frame):,} rows   {report}")
    return frame.reset_index(drop=True), report


train_frame, train_cleaning = clean_frame(train_frame, "train")
test_frame, test_cleaning = clean_frame(test_frame, "test")

if TRAIN_SUBSET:
    train_frame = train_frame.sample(n=min(TRAIN_SUBSET, len(train_frame)),
                                     random_state=SEED).reset_index(drop=True)
    print(f"train subsampled to {len(train_frame):,}")
if TEST_SUBSET:
    test_frame = test_frame.sample(n=min(TEST_SUBSET, len(test_frame)),
                                   random_state=SEED).reset_index(drop=True)
    print(f"test subsampled to {len(test_frame):,}")

log_line(f"cleaning_train={train_cleaning}")
log_line(f"cleaning_test={test_cleaning}")

In [ ]:
# Class distribution and balance
for name, frame in (("train", train_frame), ("test", test_frame)):
    counts = frame[LABEL_COLUMN].value_counts().sort_index()
    total = len(frame)
    ratio = counts.min() / counts.max()
    print(f"{name}: negative(0)={counts.get(0,0):,} ({100*counts.get(0,0)/total:.2f}%)  "
          f"positive(1)={counts.get(1,0):,} ({100*counts.get(1,0)/total:.2f}%)  "
          f"balance ratio={ratio:.4f}")
    log_line(f"class_distribution_{name} neg={counts.get(0,0)} pos={counts.get(1,0)} ratio={ratio:.4f}")

# Review length distribution, in whitespace tokens
train_frame["word_count"] = train_frame[TEXT_COLUMN].str.split().str.len()
test_frame["word_count"] = test_frame[TEXT_COLUMN].str.split().str.len()
lengths = train_frame["word_count"]
print("\nreview length in words (train):")
for label, value in (("mean", lengths.mean()), ("std", lengths.std()), ("min", lengths.min()),
                     ("25%", lengths.quantile(.25)), ("median", lengths.median()),
                     ("75%", lengths.quantile(.75)), ("90%", lengths.quantile(.90)),
                     ("95%", lengths.quantile(.95)), ("99%", lengths.quantile(.99)),
                     ("max", lengths.max())):
    print(f"  {label:<7} {value:>10.1f}")

coverage = (lengths <= CONFIG["max_length"]).mean()
print(f"\nmax_length={CONFIG['max_length']} keeps {100*coverage:.2f}% of reviews whole; "
      f"{100*(1-coverage):.2f}% are truncated")
log_line(f"length_mean={lengths.mean():.1f} length_median={lengths.median():.0f} "
         f"max_length={CONFIG['max_length']} untruncated_fraction={coverage:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(lengths.clip(upper=600), bins=60, color="steelblue")
axes[0].axvline(CONFIG["max_length"], color="crimson", linestyle="--",
                label=f"max_length={CONFIG['max_length']}")
axes[0].set_title("Review length (train, clipped at 600)")
axes[0].set_xlabel("words"); axes[0].legend()

for label, colour in ((0, "indianred"), (1, "seagreen")):
    axes[1].hist(train_frame.loc[train_frame[LABEL_COLUMN] == label, "word_count"].clip(upper=600),
                 bins=60, alpha=.55, label=f"label {label}", color=colour)
axes[1].set_title("Length by class"); axes[1].set_xlabel("words"); axes[1].legend()

counts = train_frame[LABEL_COLUMN].value_counts().sort_index()
axes[2].bar(["negative (0)", "positive (1)"], counts.values, color=["indianred", "seagreen"])
axes[2].set_title("Class distribution (train)")
for i, v in enumerate(counts.values):
    axes[2].text(i, v, f"{v:,}", ha="center", va="bottom")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "data_distribution.png", dpi=150)
plt.show()

### 1.2 Text preprocessing

Lowercasing, punctuation removal, stopword removal and tokenisation, as §2.1
requires — with one deliberate exception.

**Negations are kept.** A generic stopword list removes `not`, `no`, `never`,
`isn't` and similar. For topic classification that is harmless; for *sentiment*
it is destructive, because "not good" and "good" collapse to the same token
sequence. The stopword list below therefore has negations and a few degree
words subtracted from it.

Contractions are expanded before punctuation is stripped, so `don't` becomes
`do not` and the negation survives as a word rather than being destroyed when
the apostrophe is removed.


In [ ]:
# Standard English stopwords, written out rather than downloaded so the run has
# no network dependency and the exact list is visible in the repo.
BASE_STOPWORDS = {
    "i","me","my","myself","we","our","ours","ourselves","you","your","yours",
    "yourself","yourselves","he","him","his","himself","she","her","hers",
    "herself","it","its","itself","they","them","their","theirs","themselves",
    "what","which","who","whom","this","that","these","those","am","is","are",
    "was","were","be","been","being","have","has","had","having","do","does",
    "did","doing","a","an","the","and","but","if","or","because","as","until",
    "while","of","at","by","for","with","about","against","between","into",
    "through","during","before","after","above","below","to","from","up","down",
    "in","out","on","off","then","once","here","there","when","where","why",
    "how","all","any","both","each","few","more","most","other","some","such",
    "own","same","so","than","too","s","t","can","will","just","should","now",
}

# Removing these would destroy the sentiment signal, so they stay.
SENTIMENT_CRITICAL = {
    "not","no","nor","never","none","nothing","neither","cannot","without",
    "very","really","too","but","however","although","though","only","most",
    "more","less","least","again","against",
}

STOPWORDS = BASE_STOPWORDS - SENTIMENT_CRITICAL
print(f"stopwords kept out of the text: {len(STOPWORDS)}")
print(f"stopwords deliberately retained: {len(BASE_STOPWORDS & SENTIMENT_CRITICAL)} "
      f"-> {sorted(BASE_STOPWORDS & SENTIMENT_CRITICAL)}")

CONTRACTIONS = {
    "won't": "will not", "can't": "can not", "cannot": "can not",
    "n't": " not", "'re": " are", "'s": " is", "'d": " would",
    "'ll": " will", "'ve": " have", "'m": " am",
}

NON_ALPHANUMERIC = re.compile(r"[^a-z0-9\s]")
MULTISPACE = re.compile(r"\s+")


def preprocess(text):
    """Lowercase, expand contractions, strip punctuation, drop stopwords.

    Contractions are expanded BEFORE punctuation removal so that "don't"
    becomes "do not" rather than "dont", keeping the negation as a token the
    model can learn from.
    """
    text = text.lower()
    text = text.replace("\\n", " ").replace("\\\"", " ")
    for contraction, expansion in CONTRACTIONS.items():
        text = text.replace(contraction, expansion)
    text = NON_ALPHANUMERIC.sub(" ", text)
    text = MULTISPACE.sub(" ", text).strip()
    return [token for token in text.split() if token not in STOPWORDS]


example = train_frame[TEXT_COLUMN].iloc[0]
print("\nBEFORE:", example[:220])
print("\nAFTER :", " ".join(preprocess(example))[:220])

negation_example = "The food was not good and I would n't come back. Service wasn't terrible though."
print("\nnegation check:")
print("  raw   :", negation_example)
print("  tokens:", preprocess(negation_example))

In [ ]:
start = time.perf_counter()
train_tokens = [preprocess(t) for t in train_frame[TEXT_COLUMN]]
test_tokens = [preprocess(t) for t in test_frame[TEXT_COLUMN]]
log_line(f"tokenised train={len(train_tokens)} test={len(test_tokens)} "
         f"seconds={time.perf_counter()-start:.1f}")

# Stratified train/validation split of the OFFICIAL training split.
labels_all = train_frame[LABEL_COLUMN].to_numpy()
split_rng = np.random.default_rng(SEED)
validation_mask = np.zeros(len(labels_all), dtype=bool)
for class_value in (0, 1):
    class_indices = np.where(labels_all == class_value)[0]
    split_rng.shuffle(class_indices)
    take = int(round(len(class_indices) * CONFIG["validation_fraction"]))
    validation_mask[class_indices[:take]] = True

train_indices = np.where(~validation_mask)[0]
validation_indices = np.where(validation_mask)[0]
assert not (set(train_indices.tolist()) & set(validation_indices.tolist()))

print(f"train      : {len(train_indices):,}  positive fraction {labels_all[train_indices].mean():.4f}")
print(f"validation : {len(validation_indices):,}  positive fraction {labels_all[validation_indices].mean():.4f}")
print(f"test       : {len(test_frame):,}  positive fraction {test_frame[LABEL_COLUMN].mean():.4f}")
log_line(f"split train={len(train_indices)} validation={len(validation_indices)} test={len(test_frame)}")

### 1.3 Vocabulary and embeddings learned from scratch

The vocabulary is built from the **training portion only**. Including validation
or test text would leak information: the model would know which words exist in
data it is meant to be evaluated on.

Two reserved indices: `0` is padding, masked out everywhere so it never
contributes to a pooled representation, and `1` is the unknown token for words
below the frequency threshold or absent entirely.

No pretrained vectors are used anywhere. Each model initialises its own
embedding table randomly and learns it during training, which is what the brief
requires.


In [ ]:
frequency = Counter()
for tokens in (train_tokens[i] for i in train_indices):
    frequency.update(tokens)

eligible = [(word, count) for word, count in frequency.most_common()
            if count >= CONFIG["min_frequency"]]
kept = eligible[: CONFIG["vocab_size"] - 2]          # leave room for PAD and UNK

word_to_index = {"<pad>": PAD_INDEX, "<unk>": UNK_INDEX}
for word, _ in kept:
    word_to_index[word] = len(word_to_index)
index_to_word = {index: word for word, index in word_to_index.items()}
VOCAB_SIZE = len(word_to_index)

total_tokens = sum(frequency.values())
covered = sum(count for word, count in frequency.items() if word in word_to_index)
print(f"distinct words in training text : {len(frequency):,}")
print(f"words at frequency >= {CONFIG['min_frequency']}        : {len(eligible):,}")
print(f"vocabulary size (incl. PAD/UNK)  : {VOCAB_SIZE:,}")
print(f"token coverage                   : {100*covered/total_tokens:.2f}%  "
      f"(the rest map to <unk>)")
print("\n15 most frequent retained words:", [w for w, _ in kept[:15]])
log_line(f"vocab_size={VOCAB_SIZE} coverage={covered/total_tokens:.4f} distinct={len(frequency)}")


def encode(tokens, max_length):
    """Map tokens to indices, truncate, and pad to a fixed length."""
    indices = [word_to_index.get(token, UNK_INDEX) for token in tokens[:max_length]]
    length = len(indices)
    indices.extend([PAD_INDEX] * (max_length - length))
    return indices, max(length, 1)


MAX_LENGTH = CONFIG["max_length"]


class ReviewDataset(Dataset):
    """Encoded reviews with their true (pre-padding) lengths and labels."""

    def __init__(self, token_lists, labels):
        encoded = [encode(tokens, MAX_LENGTH) for tokens in token_lists]
        self.inputs = torch.tensor([e[0] for e in encoded], dtype=torch.long)
        self.lengths = torch.tensor([e[1] for e in encoded], dtype=torch.long)
        self.labels = torch.tensor(np.asarray(labels), dtype=torch.float32)

    def __len__(self):
        return self.inputs.shape[0]

    def __getitem__(self, index):
        return self.inputs[index], self.lengths[index], self.labels[index]


train_dataset = ReviewDataset([train_tokens[i] for i in train_indices], labels_all[train_indices])
validation_dataset = ReviewDataset([train_tokens[i] for i in validation_indices], labels_all[validation_indices])
test_dataset = ReviewDataset(test_tokens, test_frame[LABEL_COLUMN].to_numpy())

NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
loader_generator = torch.Generator(); loader_generator.manual_seed(SEED)
make_loader = lambda ds, shuffle: DataLoader(
    ds, batch_size=CONFIG["batch_size"], shuffle=shuffle, num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda", generator=loader_generator if shuffle else None)

train_loader = make_loader(train_dataset, True)
validation_loader = make_loader(validation_dataset, False)
test_loader = make_loader(test_dataset, False)

inputs_example, lengths_example, labels_example = next(iter(train_loader))
print(f"\nbatch inputs {tuple(inputs_example.shape)}  lengths {tuple(lengths_example.shape)}  "
      f"labels {tuple(labels_example.shape)}")
print(f"batches: train {len(train_loader):,}  validation {len(validation_loader):,}  test {len(test_loader):,}")

np.save(PROCESSED_DIR / "train_indices.npy", train_indices)
np.save(PROCESSED_DIR / "validation_indices.npy", validation_indices)
with (PROCESSED_DIR / "vocabulary.json").open("w", encoding="utf-8") as handle:
    json.dump({"vocab_size": VOCAB_SIZE, "word_to_index": word_to_index}, handle)

## 2. Models

All three learn their own embedding table from scratch. What differs is **how a
variable-length sequence of word vectors is reduced to one fixed vector** before
classification — that reduction is the interesting design axis here.

### `baseline_max_pool`
Takes the maximum over positions per embedding dimension. Each dimension ends up
reporting the strongest evidence anywhere in the review, which suits sentiment:
one emphatic phrase often decides the label regardless of length. Chosen over
mean pooling — the other member's baseline — because averaging dilutes a short
decisive phrase across a long review.

### `experiment_lstm`
Two stacked unidirectional LSTM layers; the final hidden state is the
representation. Unlike pooling, this is order-sensitive, so "not good" and "good
not" differ. Deliberately **unidirectional**, where the other member's model is a
BiGRU, so neither the direction structure nor the cell type overlaps.

### `experiment_attention_pool`
Learns a scalar relevance score per token, softmaxes it into weights, and takes a
weighted sum. Max pooling keeps one position per dimension and mean pooling
weights all positions equally; this learns the weighting instead, and the weights
can be inspected to see which words drove a prediction.

**Masking.** Padding is index 0 and must contribute nothing. Max pooling sets pad
positions to `-inf` before the max, attention sets their scores to `-inf` before
the softmax, and the LSTM uses packed sequences so it never reads past the real
end. Skipping this lets padding shift predictions and the bug is invisible in the
loss curve.


In [ ]:
class MaxPoolClassifier(nn.Module):
    """Embedding -> masked max pooling over positions -> MLP."""

    def __init__(self, vocab_size, embedding_dim=96, hidden_dim=128, dropout=0.10):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_INDEX)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, 1),
        )

    def forward(self, inputs, lengths):
        embedded = self.embedding(inputs)                    # (B, L, E)
        pad_mask = (inputs == PAD_INDEX).unsqueeze(-1)
        # -inf at padding so it can never win the max.
        embedded = embedded.masked_fill(pad_mask, float("-inf"))
        pooled = embedded.max(dim=1).values
        # A row that is all padding would be -inf everywhere; guard it.
        pooled = torch.nan_to_num(pooled, neginf=0.0)
        return self.classifier(self.dropout(pooled)).squeeze(-1)


class LSTMClassifier(nn.Module):
    """Embedding -> 2-layer unidirectional LSTM -> MLP on the final hidden state."""

    def __init__(self, vocab_size, embedding_dim=160, hidden_dim=128,
                 num_layers=2, dropout=0.30):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_INDEX)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=False,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, 1),
        )

    def forward(self, inputs, lengths):
        embedded = self.embedding(inputs)
        # Packing stops the LSTM from consuming padding, so the final hidden
        # state corresponds to the last REAL token rather than the last pad.
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        final_hidden = hidden[-1]                            # top layer
        return self.classifier(self.dropout(final_hidden)).squeeze(-1)


class AttentionPoolClassifier(nn.Module):
    """Embedding -> learned scalar attention pooling -> MLP."""

    def __init__(self, vocab_size, embedding_dim=128, attention_dim=64,
                 hidden_dim=128, dropout=0.20):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_INDEX)
        # One scalar score per token: tanh projection then a learned direction.
        self.attention = nn.Sequential(
            nn.Linear(embedding_dim, attention_dim), nn.Tanh(),
            nn.Linear(attention_dim, 1),
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, 1),
        )

    def forward(self, inputs, lengths, return_attention=False):
        embedded = self.embedding(inputs)                    # (B, L, E)
        scores = self.attention(embedded).squeeze(-1)        # (B, L)
        scores = scores.masked_fill(inputs == PAD_INDEX, float("-inf"))
        weights = F.softmax(scores, dim=1)
        pooled = torch.bmm(weights.unsqueeze(1), embedded).squeeze(1)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        if return_attention:
            return logits, weights
        return logits


MODEL_BUILDERS = {
    "baseline_max_pool": lambda: MaxPoolClassifier(VOCAB_SIZE, 96, 128, 0.10),
    "experiment_lstm": lambda: LSTMClassifier(VOCAB_SIZE, 160, 128, 2, 0.30),
    "experiment_attention_pool": lambda: AttentionPoolClassifier(VOCAB_SIZE, 128, 64, 128, 0.20),
}
BASELINE_NAME = "baseline_max_pool"

print(f"{'model':<28} {'parameters':>12}")
print("-" * 42)
for name, builder in MODEL_BUILDERS.items():
    count = sum(p.numel() for p in builder().parameters() if p.requires_grad)
    print(f"{name:<28} {count:>12,}")
    log_line(f"model={name} parameters={count}")

# Masking check: padding must not change a prediction.
probe = MODEL_BUILDERS[BASELINE_NAME]().to(DEVICE).eval()
with torch.no_grad():
    short = inputs_example[:4, :40].to(DEVICE)
    short_lengths = (short != PAD_INDEX).sum(1).clamp(min=1)
    padded = torch.cat([short, torch.zeros(4, 80, dtype=torch.long, device=DEVICE)], dim=1)
    difference = (probe(short, short_lengths) - probe(padded, short_lengths)).abs().max().item()
print(f"\nextra padding changed the max-pool prediction by: {difference:.3e}  <- must be 0.0")
assert difference < 1e-6, "padding is leaking into the pooled representation"
print("masking verified.")

## 3. Training

One loop for all three models so the only difference between them is the
architecture, never the training procedure. AdamW with gradient clipping, and
the checkpoint kept is the epoch with the best validation loss rather than the
last — the last epoch is often past the point of best generalisation.

`BCEWithLogitsLoss` is used instead of a sigmoid followed by `BCELoss`: it is the
same function but computed in log-space, which avoids overflow when logits are
large.


In [ ]:
@torch.no_grad()
def predict(model, loader):
    """Return (probabilities, labels) over a loader."""
    model.eval()
    probabilities, labels = [], []
    for inputs, lengths, batch_labels in loader:
        logits = model(inputs.to(DEVICE), lengths.to(DEVICE))
        probabilities.append(torch.sigmoid(logits).float().cpu().numpy())
        labels.append(batch_labels.numpy())
    return np.concatenate(probabilities), np.concatenate(labels)


def reset_peak_memory():
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()


def peak_memory_gb():
    if DEVICE.type == "cuda":
        return torch.cuda.max_memory_allocated() / 1024 ** 3
    if DEVICE.type == "mps":
        return torch.mps.driver_allocated_memory() / 1024 ** 3
    return 0.0


def train_model(name, builder):
    """Train one model and return it with its training record."""
    torch.manual_seed(SEED)          # identical initialisation conditions per model
    np.random.seed(SEED)
    random.seed(SEED)

    model = builder().to(DEVICE)
    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"],
                                  weight_decay=CONFIG["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()

    record = {"name": name, "parameter_count": parameter_count,
              "train_loss": [], "validation_loss": [], "validation_accuracy": [],
              "epoch_seconds": [], "gradient_norm_mean": []}
    best_validation_loss, best_state = float("inf"), None
    examples_seen = 0

    reset_peak_memory()
    start = time.perf_counter()
    log_line(f"TRAIN START model={name} parameters={parameter_count}")

    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        epoch_start = time.perf_counter()
        epoch_loss, batches, gradient_norms = 0.0, 0, []

        for inputs, lengths, labels in train_loader:
            inputs, lengths, labels = inputs.to(DEVICE), lengths.to(DEVICE), labels.to(DEVICE)
            logits = model(inputs, lengths)
            loss = criterion(logits, labels)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            gradient_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), CONFIG["gradient_clip"]).item()
            optimizer.step()

            epoch_loss += loss.item(); batches += 1
            gradient_norms.append(gradient_norm)
            examples_seen += labels.numel()

        validation_probabilities, validation_labels = predict(model, validation_loader)
        validation_loss = F.binary_cross_entropy(
            torch.tensor(validation_probabilities).clamp(1e-7, 1 - 1e-7),
            torch.tensor(validation_labels)).item()
        validation_accuracy = accuracy_score(validation_labels, validation_probabilities >= 0.5)
        epoch_seconds = time.perf_counter() - epoch_start

        record["train_loss"].append(epoch_loss / batches)
        record["validation_loss"].append(validation_loss)
        record["validation_accuracy"].append(validation_accuracy)
        record["epoch_seconds"].append(epoch_seconds)
        record["gradient_norm_mean"].append(float(np.mean(gradient_norms)))

        log_line(f"model={name} epoch {epoch}/{CONFIG['epochs']} "
                 f"train_loss={epoch_loss/batches:.4f} val_loss={validation_loss:.4f} "
                 f"val_acc={validation_accuracy:.4f} grad={np.mean(gradient_norms):.4f} "
                 f"seconds={epoch_seconds:.1f}")

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            record["best_epoch"] = epoch

    record["training_time_seconds"] = time.perf_counter() - start
    record["examples_per_second"] = examples_seen / record["training_time_seconds"]
    record["peak_memory_gb"] = peak_memory_gb()
    record["best_validation_loss"] = best_validation_loss

    model.load_state_dict(best_state)
    torch.save({"model": best_state, "config": CONFIG, "seed": SEED,
                "vocab_size": VOCAB_SIZE, "record": record},
               CHECKPOINT_DIR / f"{name}.pt")
    log_line(f"TRAIN DONE model={name} seconds={record['training_time_seconds']:.1f} "
             f"best_epoch={record.get('best_epoch')} examples_per_sec={record['examples_per_second']:.0f} "
             f"peak_gb={record['peak_memory_gb']:.3f}")
    return model, record


trained_models, training_records = {}, {}
for name, builder in MODEL_BUILDERS.items():
    trained_models[name], training_records[name] = train_model(name, builder)

with (OUTPUT_DIR / "training_records.json").open("w") as handle:
    json.dump(training_records, handle, indent=2)

## 4. Evaluation

Every metric the brief lists, computed on the held-out **test** split.

**ROC-AUC vs PR-AUC** — ROC-AUC asks how well the two classes are ranked apart.
PR-AUC focuses on the positive class and is the more honest figure under
imbalance. Both are reported since the brief asks for both.

**Brier score and expected calibration error** measure whether the probabilities
*mean* anything, not just whether the ranking is right. A model can be accurate
and badly calibrated — confident at 0.99 when it is right 80% of the time. Brier
is the mean squared error of the probabilities; ECE bins predictions by
confidence and averages the gap between confidence and accuracy in each bin.

**Bootstrap confidence intervals** say how much of a difference between models is
real rather than sampling noise. **McNemar's test** compares two models on the
*same* items, looking only at cases where exactly one was right — the paired
comparison the brief asks for.


In [ ]:
def expected_calibration_error(labels, probabilities, n_bins=10):
    """Average gap between confidence and accuracy, weighted by bin population."""
    confidence = np.where(probabilities >= 0.5, probabilities, 1 - probabilities)
    predictions = (probabilities >= 0.5).astype(int)
    correct = (predictions == labels).astype(float)
    edges = np.linspace(0.5, 1.0, n_bins + 1)
    error, total = 0.0, len(labels)
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (confidence > low) & (confidence <= high)
        if in_bin.sum() == 0:
            continue
        error += (in_bin.sum() / total) * abs(correct[in_bin].mean() - confidence[in_bin].mean())
    return float(error)


def bootstrap_interval(labels, probabilities, metric_function, n_resamples=1000, alpha=0.05):
    """Percentile bootstrap CI, resampling test items with replacement."""
    rng = np.random.default_rng(SEED)
    n = len(labels)
    values = []
    for _ in range(n_resamples):
        idx = rng.integers(0, n, n)
        if len(np.unique(labels[idx])) < 2:      # degenerate resample
            continue
        values.append(metric_function(labels[idx], probabilities[idx]))
    lower, upper = np.percentile(values, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lower), float(upper)


def mcnemar_test(labels, probabilities_a, probabilities_b):
    """Exact McNemar on the discordant pairs: items exactly one model got right."""
    correct_a = (probabilities_a >= 0.5).astype(int) == labels
    correct_b = (probabilities_b >= 0.5).astype(int) == labels
    b = int(np.sum(correct_a & ~correct_b))     # a right, b wrong
    c = int(np.sum(~correct_a & correct_b))     # b right, a wrong
    if b + c == 0:
        return {"b": b, "c": c, "statistic": 0.0, "p_value": 1.0}
    from scipy.stats import binomtest
    p_value = binomtest(b, b + c, 0.5).pvalue
    statistic = (abs(b - c) - 1) ** 2 / (b + c)   # continuity-corrected chi-square
    return {"b": b, "c": c, "statistic": float(statistic), "p_value": float(p_value)}


# Slices for the robustness metric: review length, which is the dimension most
# likely to interact with pooling and with the 320-token truncation.
test_word_counts = test_frame["word_count"].to_numpy()
if TEST_SUBSET or len(test_word_counts) != len(test_dataset):
    test_word_counts = test_word_counts[: len(test_dataset)]
short_cut, long_cut = np.percentile(test_word_counts, [33, 66])
SLICES = {
    f"short (<={short_cut:.0f} words)": test_word_counts <= short_cut,
    f"medium ({short_cut:.0f}-{long_cut:.0f})": (test_word_counts > short_cut) & (test_word_counts <= long_cut),
    f"long (>{long_cut:.0f} words)": test_word_counts > long_cut,
    f"truncated (>{CONFIG['max_length']} words)": test_word_counts > CONFIG["max_length"],
}
print("test slices:")
for name, mask in SLICES.items():
    print(f"  {name:<32} n={mask.sum():>7,}  positive fraction={test_frame[LABEL_COLUMN].to_numpy()[:len(mask)][mask].mean():.3f}")

In [ ]:
test_predictions = {}
for name, model in trained_models.items():
    probabilities, labels = predict(model, test_loader)
    test_predictions[name] = probabilities
test_labels = labels            # identical across models: same loader, no shuffle

all_metrics = {}
for name, probabilities in test_predictions.items():
    predicted = (probabilities >= 0.5).astype(int)
    record = training_records[name]

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        test_labels, predicted, average="macro", zero_division=0)
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        test_labels, predicted, average="micro", zero_division=0)
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        test_labels, predicted, average="weighted", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(test_labels, predicted, labels=[0, 1]).ravel()

    accuracy_ci = bootstrap_interval(test_labels, probabilities,
                                     lambda y, p: accuracy_score(y, p >= 0.5))
    f1_ci = bootstrap_interval(test_labels, probabilities,
                               lambda y, p: precision_recall_fscore_support(
                                   y, p >= 0.5, average="macro", zero_division=0)[2])
    mcc_ci = bootstrap_interval(test_labels, probabilities,
                                lambda y, p: matthews_corrcoef(y, p >= 0.5))

    metrics = {
        "model": name,
        "accuracy": accuracy_score(test_labels, predicted),
        "precision_macro": precision_macro, "recall_macro": recall_macro, "f1_macro": f1_macro,
        "precision_micro": precision_micro, "recall_micro": recall_micro, "f1_micro": f1_micro,
        "precision_weighted": precision_weighted, "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "confusion_tn": int(tn), "confusion_fp": int(fp),
        "confusion_fn": int(fn), "confusion_tp": int(tp),
        "roc_auc": roc_auc_score(test_labels, probabilities),
        "pr_auc": average_precision_score(test_labels, probabilities),
        "mcc": matthews_corrcoef(test_labels, predicted),
        "brier_score": brier_score_loss(test_labels, probabilities),
        "expected_calibration_error": expected_calibration_error(test_labels, probabilities),
        "accuracy_ci95_low": accuracy_ci[0], "accuracy_ci95_high": accuracy_ci[1],
        "f1_macro_ci95_low": f1_ci[0], "f1_macro_ci95_high": f1_ci[1],
        "mcc_ci95_low": mcc_ci[0], "mcc_ci95_high": mcc_ci[1],
        "parameter_count": record["parameter_count"],
        "training_time_sec": record["training_time_seconds"],
        "examples_per_sec": record["examples_per_second"],
        "peak_memory_gb": record["peak_memory_gb"],
        "best_epoch": record.get("best_epoch"),
        "device_name": DEVICE_NAME,
    }

    # Paired McNemar against the baseline.
    if name == BASELINE_NAME:
        metrics["mcnemar_statistic"], metrics["mcnemar_p_value"] = float("nan"), float("nan")
        metrics["mcnemar_vs"] = "(is the baseline)"
    else:
        test_result = mcnemar_test(test_labels, test_predictions[BASELINE_NAME], probabilities)
        metrics["mcnemar_statistic"] = test_result["statistic"]
        metrics["mcnemar_p_value"] = test_result["p_value"]
        metrics["mcnemar_baseline_only_correct"] = test_result["b"]
        metrics["mcnemar_model_only_correct"] = test_result["c"]
        metrics["mcnemar_vs"] = BASELINE_NAME

    # Per-slice macro-F1 and error rate.
    for slice_name, mask in SLICES.items():
        if mask.sum() == 0:
            continue
        slice_labels, slice_predicted = test_labels[mask], predicted[mask]
        key = slice_name.split(" ")[0]
        if len(np.unique(slice_labels)) < 2:
            metrics[f"slice_{key}_macro_f1"] = float("nan")
        else:
            metrics[f"slice_{key}_macro_f1"] = precision_recall_fscore_support(
                slice_labels, slice_predicted, average="macro", zero_division=0)[2]
        metrics[f"slice_{key}_error_rate"] = float((slice_labels != slice_predicted).mean())
        metrics[f"slice_{key}_n"] = int(mask.sum())

    all_metrics[name] = metrics
    log_line(f"TEST model={name} acc={metrics['accuracy']:.4f} f1_macro={metrics['f1_macro']:.4f} "
             f"roc_auc={metrics['roc_auc']:.4f} mcc={metrics['mcc']:.4f} "
             f"brier={metrics['brier_score']:.4f} ece={metrics['expected_calibration_error']:.4f}")

metrics_frame = pd.DataFrame(all_metrics).T
pd.set_option("display.width", 200)
print(metrics_frame[["accuracy", "f1_macro", "roc_auc", "pr_auc", "mcc",
                     "brier_score", "expected_calibration_error",
                     "parameter_count", "training_time_sec"]].to_string())

metrics_frame.to_csv(MEMBER_ROOT / "metrics_report.csv", index=False)
with (OUTPUT_DIR / "metrics.json").open("w") as handle:
    json.dump(all_metrics, handle, indent=2, default=str)
print("\nwritten:", MEMBER_ROOT / "metrics_report.csv")

In [ ]:
# Confusion matrices, ROC/PR curves and calibration plots.
fig, axes = plt.subplots(1, len(all_metrics), figsize=(5 * len(all_metrics), 4.2))
for ax, (name, probabilities) in zip(np.atleast_1d(axes), test_predictions.items()):
    cm = confusion_matrix(test_labels, probabilities >= 0.5, labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_title(f"{name}\nacc={all_metrics[name]['accuracy']:.4f}")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["pred 0", "pred 1"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["true 0", "true 1"])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices" / "confusion_matrices.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for name, probabilities in test_predictions.items():
    fpr, tpr, _ = roc_curve(test_labels, probabilities)
    axes[0].plot(fpr, tpr, label=f"{name} ({all_metrics[name]['roc_auc']:.4f})")
    precision_curve, recall_curve, _ = precision_recall_curve(test_labels, probabilities)
    axes[1].plot(recall_curve, precision_curve, label=f"{name} ({all_metrics[name]['pr_auc']:.4f})")

    # Reliability curve: predicted confidence vs observed accuracy.
    edges = np.linspace(0, 1, 11)
    centres, observed = [], []
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (probabilities > low) & (probabilities <= high)
        if in_bin.sum() > 0:
            centres.append(probabilities[in_bin].mean())
            observed.append(test_labels[in_bin].mean())
    axes[2].plot(centres, observed, marker="o", label=f"{name} (ECE {all_metrics[name]['expected_calibration_error']:.4f})")

axes[0].plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
axes[0].set_title("ROC"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend()
axes[1].set_title("Precision-Recall"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend()
axes[2].plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
axes[2].set_title("Calibration (reliability)"); axes[2].set_xlabel("Mean predicted probability")
axes[2].set_ylabel("Observed positive rate"); axes[2].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "roc_pr_calibration.png", dpi=150)
plt.show()

## 5. Error review

§2.2.4 requires manual review of **20 errors** from your own best model: 5
confident false positives, 5 confident false negatives, 5 near-threshold errors,
and 5 slice-specific failures. Each needs an error type and one testable fix.

The cell below selects and exports the 20 cases with their text and predicted
probability. **The error type and the fix are yours to write** — that is the
analysis the brief asks for, and the viva will ask about it.


In [ ]:
best_model_name = max(all_metrics, key=lambda n: all_metrics[n]["f1_macro"])
probabilities = test_predictions[best_model_name]
predicted = (probabilities >= 0.5).astype(int)
print(f"best model by macro-F1: {best_model_name} ({all_metrics[best_model_name]['f1_macro']:.4f})")

texts = test_frame[TEXT_COLUMN].to_numpy()[: len(test_labels)]
word_counts = test_word_counts[: len(test_labels)]
wrong = np.where(predicted != test_labels)[0]
print(f"total errors: {len(wrong):,} of {len(test_labels):,} ({100*len(wrong)/len(test_labels):.2f}%)")

false_positives = wrong[test_labels[wrong] == 0]
false_negatives = wrong[test_labels[wrong] == 1]

confident_fp = false_positives[np.argsort(-probabilities[false_positives])][:5]
confident_fn = false_negatives[np.argsort(probabilities[false_negatives])][:5]
near_threshold = wrong[np.argsort(np.abs(probabilities[wrong] - 0.5))][:5]

longest_slice = SLICES[[k for k in SLICES if k.startswith("long")][0]][: len(test_labels)]
slice_errors = wrong[longest_slice[wrong]]
slice_specific = slice_errors[np.argsort(-np.abs(probabilities[slice_errors] - 0.5))][:5]

groups = [("confident_false_positive", confident_fp),
          ("confident_false_negative", confident_fn),
          ("near_threshold", near_threshold),
          ("slice_long_review", slice_specific)]

rows = []
for category, indices in groups:
    for index in indices:
        rows.append({
            "category": category,
            "test_index": int(index),
            "true_label": int(test_labels[index]),
            "predicted_label": int(predicted[index]),
            "p_positive": float(probabilities[index]),
            "word_count": int(word_counts[index]),
            "review_excerpt": texts[index][:400].replace("\n", " "),
            "error_type": "",        # <- fill in
            "testable_fix": "",      # <- fill in
        })

error_frame = pd.DataFrame(rows)
error_frame.to_csv(OUTPUT_DIR / "error_review_20.csv", index=False)

for category, _ in groups:
    subset = error_frame[error_frame["category"] == category]
    print(f"\n{'=' * 78}\n{category.upper()}  ({len(subset)} cases)\n{'=' * 78}")
    for _, row in subset.iterrows():
        print(f"\n  true={row['true_label']} predicted={row['predicted_label']} "
              f"p(pos)={row['p_positive']:.4f} words={row['word_count']}")
        print(f"  {row['review_excerpt'][:300]}")

print(f"\n\nwritten: {OUTPUT_DIR / 'error_review_20.csv'}")
print("Fill in error_type and testable_fix yourself, then copy into failure_analysis.md")
log_line(f"error_review best_model={best_model_name} total_errors={len(wrong)} exported=20")

## 6. Comparative analysis

§2.3 asks you to compare your three models, then compare against your teammate's
as a team, and to discuss strengths, weaknesses, limitations and future work.

The numbers are above; the argument is yours to write in `results.md`. Things
worth looking at before you write:

- **Does the ranking survive its confidence interval?** If two models' accuracy
  CIs overlap, "A beats B" is not supported by this test set. The bootstrap
  intervals are in the metrics table.
- **What does McNemar say?** It compares the models on the same items. A small
  p-value means the difference in *which* items they get right is systematic,
  not noise. Two models can have near-identical accuracy and still disagree on a
  large number of individual reviews.
- **Does the LSTM earn its cost?** It has the most parameters and the slowest
  training. If it does not clearly beat max pooling, that is a real finding worth
  stating, not a failure to hide.
- **Where does truncation hurt?** The `truncated` slice holds reviews longer than
  320 tokens. Compare its error rate against the short slice: if it is much
  worse, `max_length` is a limitation to name.
- **Is anything badly calibrated?** A model can rank well (high ROC-AUC) and
  still output probabilities that mean nothing (high ECE). Say which, and why it
  matters for a deployed classifier.
